A notebook for trying to cluster chunks by taking their embeddings, using UMAP for dimensionality reduction and then plotting.

In [1]:
from ddd_policy_tracer.analysis.chunks.adapters import SQLiteDocumentChunkRepository
import logging

from ddd_policy_tracer.utils.db_manager import DbManager

logging.basicConfig(level=logging.INFO)
db_manager = DbManager.get_instance("data/new2.db")
chunk_repo = SQLiteDocumentChunkRepository(db_manager=db_manager)

/Users/thomasscelsi/Documents/programming/ddd-policy-tracer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd

chunks = await chunk_repo.list_chunks(source_id="hansard")

df = pd.DataFrame([{
    "chunk_id": chunk.chunk_id,
    "source_id": chunk.source_id,
    "source_document_id": chunk.source_document_id,
    "text": chunk.chunk_text,
    "embedding": chunk.embedding,
    "topics": chunk.topics,
    "metadata": chunk.metadata["doc_metadata"],
} for chunk in chunks])
df['chunk_length'] = df['text'].apply(len)
df.columns

Index(['chunk_id', 'source_id', 'source_document_id', 'text', 'embedding',
       'topics', 'metadata', 'chunk_length'],
      dtype='str')

In [3]:
df = pd.concat([df.drop(columns=['metadata']), pd.json_normalize(df['metadata'])], axis=1)

In [4]:
df.describe()

,chunk_length
count,15027.000000
mean,699.899714
std,555.910490
min,2.000000
25%,118.000000
50%,753.000000
75%,1230.000000
max,9504.000000


In [5]:
# filter df by text length
has_embedding_mask = df['embedding'].notnull()
has_topics_mask = df['topics'].notnull()
mask = has_embedding_mask & has_topics_mask

filtered_df = df[mask]
filtered_df = filtered_df.reset_index(drop=True)
filtered_df.describe()

,chunk_length
count,8397.000000
mean,1132.146005
std,253.537790
min,501.000000
25%,966.000000
50%,1191.000000
75%,1328.000000
max,2449.000000


In [69]:
import numpy as np
# sum embeddings by source_document_id
doc_embeddings = filtered_df.groupby("source_document_id").agg({
    "embedding": lambda x: np.mean(np.array(x.tolist()), axis=0),
}).reset_index()

In [70]:
import umap

mapper = umap.UMAP().fit(np.array(doc_embeddings.embedding.tolist()))

In [71]:
import textwrap
import umap.plot

hover_data = pd.DataFrame(
    {
        "index": doc_embeddings["source_document_id"],
        "label": doc_embeddings["source_document_id"],
    },
)
p = umap.plot.interactive(
    mapper,
    labels=doc_embeddings.source_document_id.to_list(),
    hover_data=hover_data,
    point_size=5,
    theme="fire",
)
umap.plot.show(p)

In [72]:
import umap

mapper = umap.UMAP().fit(list(filtered_df["embedding"]))

In [73]:
import textwrap
import umap.plot

hover_data = pd.DataFrame(
    {
        "index": filtered_df["chunk_id"],
        "source_id": filtered_df["source_id"],
        "label": filtered_df["source_document_id"],
        "text": [textwrap.wrap(text, width=50) for text in filtered_df["text"]],
    },
)
p = umap.plot.interactive(
    mapper,
    labels=filtered_df.source_document_id.to_list(),
    hover_data=hover_data,
    point_size=5,
    theme="fire",
)
umap.plot.show(p)

In [13]:
import textwrap

def visualise_texts(texts: list[str], width: int = 100):
    for text in texts:
        print('\n'.join(textwrap.wrap(text, width, break_long_words=False)))
        print('=' * width)
        print('=' * width)

visualise_texts(filtered_df.text.sample(3))

## **Grattan Institute**   The Grattan Institute is a prominent Australian research organisation,
with a wellregarded energy and climate research program. Grattan urges reform of the FTCS “to help
repair the budget and reduce carbon emissions,” noting that the policy is “out of step with the
government’s climate goals” which include “the phase-out of inefficient fossil-fuel subsidies.”[25]
## **5.1 Introduction**   The previous section outlined some of the difficulties faced by some
private schools in achieving the objectives of cultural diversity. This section focuses more
specifically on the inherent tension between the public funding of elite private schools and the
pursuit of equality of opportunity.   One of the goals specified in the _National Goals for
Schooling_ recommends that ‘all students have access to the high quality education necessary to
enable the completion of school education to Year 12’ (MCEETYA 1999). Despite the widespread
recognition of the pressures on both the 

In [17]:
long_df = df[df['chunk_length'] > 2000]

visualise_texts(long_df.sort_values('chunk_length', ascending=False).head(10).text)

## **References**   Argy, F. 2003, _Where to From Here? Australian Egalitarianism Under Threat_ ,
Allen and Unwin, Sydney.   Association of Independent Schools Victoria (AISV) 1998, ‘What Parents
Want from their Children’s Education in Independent Schools: Condensed Report’,
http://www.ais.vic.edu.au/public/docs/what_parents_want.pdf (24 May 2004).   Association of Women
Educators (AWE) 2004, ‘Current Project: Pregnancy and Parenting Students Policy’,
http://www.awe.asn.au/policy.php (8 May 2004).   Aulich, C. and Aulich, T. 2003, ‘Proposals for
Improved Accountability for Government Funding to Private Schools’, Australian School of Government
Studies, Canberra, http://www.aeufederal.org.au/Debates/Aulichgovtfund.pdf (11 June 2004).
Australian Academy of Technological Sciences and Engineering (ATSE) 2003, ‘Policy Statement On
Education’, http://www.atse.org.au/publications/policy/educn.htm (8 May 2004).   Australian
Association of Christian Schools (AACS) 2002, ‘Statement of Affirmatio

In [22]:
long_df[long_df.text.apply(lambda x: x.startswith("## **References** "))].source_document_id.to_list()

['https://australiainstitute.org.au/report/the-accountability-of-private-schools-to-public-values',
 'https://australiainstitute.org.au/report/no-blood-no-job',
 'https://australiainstitute.org.au/report/doing-it-tough',
 'https://australiainstitute.org.au/report/an-industrial-strategy-for-domestic-manufacturing-of-onshore-and-offshore-wind-energy-towers-and-equipment',
 'https://australiainstitute.org.au/report/solid-foundations-bright-future',
 'https://australiainstitute.org.au/report/reforming-university-governance-in-australia',
 'https://australiainstitute.org.au/report/australias-greenhouse-strategy-can-the-future-be-rescued']

## Chunk pertaining to "Israel" plot

In [11]:
df.columns

Index(['chunk_id', 'source_id', 'source_document_id', 'text', 'embedding',
       'topics', 'chunk_length', 'talker.id', 'talker.name',
       'talker.first_name', 'talker.last_name', 'talker.electorate',
       'talker.party'],
      dtype='str')

In [6]:
palestine_israel_mentioned = filtered_df["text"].str.lower().str.contains("palestine|israel|gaza|west bank|gaza strip|hamas|fatah|jerusalem|settlements|occupation|intifada|zionist|palestinian authority")
israel_df = filtered_df[palestine_israel_mentioned]
israel_df.reset_index(drop=True, inplace=True)
len(israel_df)

153

In [8]:
from gliner2 import GLiNER2  # requires gliner2[local]

extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
schema = extractor.create_schema().classification(
    "pro_israel_sentiment",
    ["positive", "negative", "neutral"],
)
results = []
for text in israel_df["text"].to_list():
    results.append(extractor.extract(text, schema, include_confidence=True))

INFO:httpx:HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/f5b2ecedebe4381b088c1cf276f5bf72a52cac54/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/encoder_config/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/f5b2ecedebe4381b088c1cf276f5bf72a52cac54/encoder_config%2Fconfig.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/f5b2ecedebe4381b088c1cf276f5bf72a52cac54/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://hug

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


In [9]:
israel_df['pro_israel_sentiment'] = results

In [10]:
israel_df = pd.concat([israel_df.drop(columns=['pro_israel_sentiment']), pd.json_normalize(israel_df['pro_israel_sentiment'])], axis=1)

In [11]:
import umap.plot
import umap
import numpy as np

mapper = umap.UMAP().fit(np.array(israel_df.embedding.tolist()))
hover_data = pd.DataFrame(
    {
        "label": israel_df["chunk_id"],
        "talker": israel_df["talker.name"],
        "party": israel_df["talker.party"],
        "pro_israel_sentiment": israel_df["pro_israel_sentiment.label"],
        "pro_israel_sentiment_confidence": israel_df["pro_israel_sentiment.confidence"],
    },
)
p = umap.plot.interactive(
    mapper,
    labels=israel_df["talker.party"].to_list(),
    hover_data=hover_data,
    point_size=5,
    theme="fire",
)
umap.plot.show(p)

/Users/thomasscelsi/Documents/programming/ddd-policy-tracer/.venv/lib/python3.12/site-packages/numba/np/ufunc/dufunc.py:346: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
/Users/thomasscelsi/Documents/programming/ddd-policy-tracer/.venv/lib/python3.12/site-packages/numba/np/ufunc/dufunc.py:346: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
/Users/thomasscelsi/Documents/programming/ddd-policy-tracer/.venv/lib/python3.12/site-packages/numba/np/ufunc/dufunc.py:346: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhap

In [12]:
import textwrap

def visualise_text(row: pd.Series, width: int = 100):
    print(f"Chunk ID: {row['chunk_id']}")
    print(f"Talker: {row['talker.name']} ({row['talker.party']})")
    print(f"Pro-Israel Sentiment: {row['pro_israel_sentiment.label']} ({row['pro_israel_sentiment.confidence']})")
    print('\n'.join(textwrap.wrap(row['text'], width, break_long_words=False)))
    print('=' * width)

def visualise_texts(df: pd.DataFrame, width: int = 100):
    for _, row in df.iterrows():
        visualise_text(row, width)

visualise_texts(israel_df.sample(3))

Chunk ID: chunk_bfb2daa724e8fd13
Talker: Ananda-Rajah, Michelle (ALP)
Pro-Israel Sentiment: negative (1.0)
I would be the only senator in this chamber who has actually been to Israel, boots on the ground,
shortly after October 7. I went there in December 2023, along with my colleague the member for
Macnamara and the then senators Birmingham and Fawcett, who have now departed this chamber. What I
saw shocked me to the core. It was like nothing I have ever experienced. I have seen plenty of death
as a doctor—believe me, I have—but this was a level of atrocity and depravity that I've never
experienced before and I will never forget.   I went to Kfar Aza, a kibbutz on the Gaza border. I
could see the plumes of smoke. I could hear the gunships going overhead and the bombs dropping. It
was frightening. Kfar Aza was a burnt-out shell, with pockmarked walls and blood splattered on the
walls, mattresses and furniture, and you could smell the death. I then went to a town called Sderot,
which, ag